In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
NEBIUS__MODEL="MiniMax/MiniMax-M2.5",
NEBIUS__FAST_MODEL="zai-org/GLM-4.7-FP8"
NEBIUS__DEFAULT_MODEL="Qwen/Qwen3-30B-A3B-fast",

In [18]:
from langchain_nebius import ChatNebius

model = ChatNebius(
    model="nvidia/nemotron-3-super-120b-a12b",
    temperature=0.6,
    top_p=0.95
)

In [19]:
from deepagents import create_deep_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


def calculator(expression: str) -> str:
    """Evaluate a basic math expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating expression: {e}"


In [20]:
agent = create_deep_agent(
    model=model,
    tools=[get_weather, calculator],
    system_prompt="You are a helpful assistant",
)

In [21]:
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='4e9084d7-829b-4d6c-b94f-029f5e9ddc4a'),
  AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 6212, 'total_tokens': 6277, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b', 'system_fingerprint': None, 'id': 'chatcmpl-00d728af659b47feaf2db6b31d84658b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9ee1-c85a-7db2-8a60-135241cc79b5-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'sf'}, 'id': 'chatcmpl-tool-09f1e35b6fdb41deb73b55fe331d6837', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 6212, 'output_tokens': 65, 'total_tokens': 6277, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}

In [23]:
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius

model = ChatNebius(model="nvidia/nemotron-3-super-120b-a12b",temperature=0.6,top_p=0.95)

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend("./agent-workspace", virtual_mode=True),
)

In [24]:
response = agent.invoke(
	{"messages": [{"role": "user", "content": "List all Python files in the workspace"}]}
)
print(response)

{'messages': [HumanMessage(content='List all Python files in the workspace', additional_kwargs={}, response_metadata={}, id='8fa4d4ba-39f4-4a7b-9c36-5125c9ffb927'), AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 6084, 'total_tokens': 6146, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b', 'system_fingerprint': None, 'id': 'chatcmpl-65a74c58ba2242a7b213daad6e541017', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9f26-5311-7221-b302-dab2499a5688-0', tool_calls=[{'name': 'glob', 'args': {'pattern': '**/*.py'}, 'id': 'chatcmpl-tool-900734b819624a87850d85c068aa1357', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 6084, 'output_tokens': 62, 'total_tokens': 6146, 'input_token_details': {'cache_read': 0}, 'output_token_d

In [26]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Summarize the contents of the workspace"}]}
):
    print(chunk, end="", flush=True)

{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Summarize the contents of the workspace', additional_kwargs={}, response_metadata={}, id='37bafd4f-0c32-44b3-83b5-6d31e8aa79e9')])}}{'model': {'messages': [AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 6085, 'total_tokens': 6144, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b', 'system_fingerprint': None, 'id': 'chatcmpl-dee03ab32af148ec805d300a3b79fff4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9f27-e064-7580-b037-607897dde520-0', tool_calls=[{'name': 'ls', 'args': {'path': '/'}, 'id': 'chatcmpl-tool-45b750832c5949c488a63aa904f41ecd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 6085, 'output_tokens': 59, 'tota

In [40]:
for chunk in agent.stream({"messages": [{"role": "user", "content": "Summarize the contents of the workspace"}]}):
    node = list(chunk.keys())[0]
    value = chunk[node]

    print(f"\n——— Node: {node} ———")

    if value is None:
        print("  (no data)")
        continue

    messages = value.get("messages", [])
    # `messages` can be a langgraph.types.Overwrite wrapper in stream chunks
    if hasattr(messages, "value"):
        messages = messages.value
    elif messages is None:
        messages = []
    elif not isinstance(messages, (list, tuple)):
        messages = [messages]

    for msg in messages:
        msg_type = type(msg).__name__
        print(f"  [{msg_type}]")

        # AIMessage: check for text content or tool calls
        if hasattr(msg, "content") and msg.content.strip():
            print(f"  content: {msg.content.strip()}")

        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  tool_call -> name: {tc['name']}, args: {tc['args']}")

        # ToolMessage: result of tool execution
        if hasattr(msg, "name") and hasattr(msg, "artifact"):
            print(f"  tool result -> tool: {msg.name}")
            print(f"  raw content: {msg.content}")
            if hasattr(msg.artifact, "entries"):
                for entry in msg.artifact.entries:
                    print(f"    {'DIR' if entry['is_dir'] else 'FILE'} {entry['path']}")


——— Node: PatchToolCallsMiddleware.before_agent ———
  [HumanMessage]
  content: Summarize the contents of the workspace

——— Node: model ———
  [AIMessage]
  tool_call -> name: ls, args: {'path': '/'}

——— Node: TodoListMiddleware.after_model ———
  (no data)

——— Node: tools ———
  [ToolMessage]
  content: ['/.python-version', '/dev-requirements.txt', '/examples/', '/image.png', '/main.py', '/responseprint.py']
  tool result -> tool: ls
  raw content: ['/.python-version', '/dev-requirements.txt', '/examples/', '/image.png', '/main.py', '/responseprint.py']
    FILE /.python-version
    FILE /dev-requirements.txt
    DIR /examples/
    FILE /image.png
    FILE /main.py
    FILE /responseprint.py

——— Node: model ———
  [AIMessage]
  tool_call -> name: ls, args: {'path': '/examples'}

——— Node: TodoListMiddleware.after_model ———
  (no data)

——— Node: tools ———
  [ToolMessage]
  content: ['/examples/__pycache__/', '/examples/agent.py']
  tool result -> tool: ls
  raw content: ['/examples/_

In [ ]:
# In JupyterLab, await works directly at the top level — no need for asyncio.run().

In [25]:
import asyncio

async def run_async():
    response = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "What files exist in the workspace?"}]}
    )
    print(response)

await run_async()

{'messages': [HumanMessage(content='What files exist in the workspace?', additional_kwargs={}, response_metadata={}, id='8f76e297-c4cc-4d72-8f15-94d81702b16f'), AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 6084, 'total_tokens': 6145, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b', 'system_fingerprint': None, 'id': 'chatcmpl-6dbb24f77b234b6fb04aad7f7083d523', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9f27-6493-75b2-97d8-4b7dedd63d6d-0', tool_calls=[{'name': 'ls', 'args': {'path': '/'}, 'id': 'chatcmpl-tool-499ea1c1c0444dc6853df7180f254586', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 6084, 'output_tokens': 61, 'total_tokens': 6145, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}), 

In [38]:
async def run_async_stream():
    async for chunk in agent.astream(
        {"messages": [{"role": "user", "content": "Describe the workspace structure"}]}
    ):
        print(chunk, end="", flush=True)

await run_async_stream()

{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Describe the workspace structure', additional_kwargs={}, response_metadata={}, id='a30e759b-fdb1-41b9-8f6e-bbd52809f934')])}}{'model': {'messages': [AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 6082, 'total_tokens': 6136, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b', 'system_fingerprint': None, 'id': 'chatcmpl-b5e7e2506309499e86902d123c573493', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9f3c-65fd-74d2-95fa-459beac35ca3-0', tool_calls=[{'name': 'ls', 'args': {'path': '/'}, 'id': 'chatcmpl-tool-ebc3885758f04bda9d33ba158486f624', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 6082, 'output_tokens': 54, 'total_token

In [27]:
import json

def pretty_agent_response(response: dict):
    msgs = response.get("messages", [])
    print(f"Total messages: {len(msgs)}\n")

    for i, m in enumerate(msgs, 1):
        role = type(m).__name__
        print(f"[{i}] {role}")

        # Main text
        content = getattr(m, "content", "")
        if content:
            print(f"  content: {content}")

        # Tool calls (if any)
        tool_calls = getattr(m, "tool_calls", None)
        if tool_calls:
            print("  tool_calls:")
            for tc in tool_calls:
                print(f"    - {tc.get('name')}({tc.get('args')}) id={tc.get('id')}")

        # Tool message metadata
        tool_name = getattr(m, "name", None)
        tool_call_id = getattr(m, "tool_call_id", None)
        if tool_name or tool_call_id:
            print(f"  tool: name={tool_name}, tool_call_id={tool_call_id}")

        # Token usage (if available)
        usage = getattr(m, "usage_metadata", None) or {}
        if usage:
            print(
                f"  tokens: in={usage.get('input_tokens')}, "
                f"out={usage.get('output_tokens')}, total={usage.get('total_tokens')}"
            )

        print()  # blank line between messages



def inspect_obj(obj, name="obj", depth=0, max_depth=4):
    indent = "  " * depth
    print(f"{indent}{name}: {type(obj)}")

    if depth >= max_depth:
        return

    if isinstance(obj, dict):
        for k, v in obj.items():
            inspect_obj(v, f"{name}[{k!r}]", depth + 1, max_depth)
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            inspect_obj(v, f"{name}[{i}]", depth + 1, max_depth)
    else:
        attrs = [a for a in dir(obj) if not a.startswith("_")]
        if attrs:
            print(f"{indent}  attrs: {attrs}")



In [45]:
# If you just want the final AIMessage text with no tool calls
final_answer = ""

for chunk in agent.stream({"messages": [{"role": "user", "content": "Summarize the contents of the main.py file and add the installation dependencies of langchain core in dev-requirements.txt"}]}):
    node = list(chunk.keys())[0]
    value = chunk[node]
    if value is None:
        continue
    messages = value.get("messages", [])
    if hasattr(messages, "value"):  # langgraph.types.Overwrite
        messages = messages.value
    elif messages is None:
        messages = []
    elif not isinstance(messages, (list, tuple)):
        messages = [messages]

    for msg in messages:
        if (
            type(msg).__name__ == "AIMessage"
            and msg.content.strip()
            and not getattr(msg, "tool_calls", [])
        ):
            final_answer = msg.content.strip()

print(final_answer)

**Summary of main.py:**
The `main.py` file contains a simple Python script that defines a `main()` function which prints "Hello from langchain-dev!" and then calls this function when the script is executed directly.

**Update to dev-requirements.txt:**
Added the dependency `langchain-core` to the development requirements file. The file now contains:
```
langchain-core
```


In [31]:
response = agent.invoke(
	{"messages": [{"role": "user", "content": "List all Python files in the workspace"}]}
)
print(response)

{'messages': [HumanMessage(content='List all Python files in the workspace', additional_kwargs={}, response_metadata={}, id='066e45f1-dfb3-4d4e-af7b-204f3ac049af'), AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 6084, 'total_tokens': 6141, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b', 'system_fingerprint': None, 'id': 'chatcmpl-82240d19095c4e719fe18f9e3634cdad', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9f38-3715-73f0-8e90-abe4258d3a69-0', tool_calls=[{'name': 'glob', 'args': {'pattern': '**/*.py'}, 'id': 'chatcmpl-tool-58b9abcb5dea40e8a839ee8c248c51fc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 6084, 'output_tokens': 57, 'total_tokens': 6141, 'input_token_details': {'cache_read': 0}, 'output_token_d

In [32]:
inspect_obj(response)

obj: <class 'dict'>
  obj['messages']: <class 'list'>
    obj['messages'][0]: <class 'langchain_core.messages.human.HumanMessage'>
      attrs: ['additional_kwargs', 'construct', 'content', 'content_blocks', 'copy', 'dict', 'from_orm', 'get_lc_namespace', 'id', 'is_lc_serializable', 'json', 'lc_attributes', 'lc_id', 'lc_secrets', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'name', 'parse_file', 'parse_obj', 'parse_raw', 'pretty_print', 'pretty_repr', 'response_metadata', 'schema', 'schema_json', 'text', 'to_json', 'to_json_not_implemented', 'type', 'update_forward_refs', 'validate']
    obj['messages'][1]: <class 'langchain_core.messages.ai.AIMessage'>
      attrs: ['additional_kwargs', 'construct', 'content', 'content_blocks

In [34]:
pretty_agent_response(response)

Total messages: 4

[1] HumanMessage
  content: List all Python files in the workspace

[2] AIMessage
  content: 


  tool_calls:
    - glob({'pattern': '**/*.py'}) id=chatcmpl-tool-58b9abcb5dea40e8a839ee8c248c51fc
  tokens: in=6084, out=57, total=6141

[3] ToolMessage
  content: ['/main.py', '/responseprint.py']
  tool: name=glob, tool_call_id=chatcmpl-tool-58b9abcb5dea40e8a839ee8c248c51fc

[4] AIMessage
  content: 
Here are all Python files in the workspace:

- `/main.py`
- `/responseprint.py`
  tokens: in=6136, out=88, total=6224



In [36]:
# was: response.content['messages'][-1]
response["messages"][-1]

AIMessage(content='\nHere are all Python files in the workspace:\n\n- `/main.py`\n- `/responseprint.py`', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 6136, 'total_tokens': 6224, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b', 'system_fingerprint': None, 'id': 'chatcmpl-f75f6cb4785e45beadc29e5cefd86483', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9f38-4c68-7932-b9b0-34c1930ff97e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6136, 'output_tokens': 88, 'total_tokens': 6224, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

In [37]:
response["messages"][-1].content

'\nHere are all Python files in the workspace:\n\n- `/main.py`\n- `/responseprint.py`'